<a href="https://colab.research.google.com/github/dalmasm/AI/blob/Data-Science/DPO_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

In [2]:
pip install --upgrade transformers huggingface_hub

In [3]:
pip install bitsandbytes accelerate

In [4]:
pip install datasets

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Choose a model. For example, a smaller version of Gemma:
model_name = "google/gemma-2b-it" # Or "mistralai/Mistral-7B-v0.1"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model (quantization helps fit larger models in memory)
# You might need to install 'bitsandbytes' and 'accelerate' for 4-bit loading

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16, # Use bfloat16 for memory efficiency if your GPU supports it
    load_in_4bit=True # Load in 4-bit for even more memory saving, though it can impact speed
)

# Now 'model' and 'tokenizer' are ready to be used!
print(f"Model {model_name} loaded successfully!")
# You can quickly test it:
# inputs = tokenizer("Tell me a short story about a brave mouse:", return_tensors="pt")
# outputs = model.generate(**inputs, max_new_tokens=50)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model google/gemma-2b-it loaded successfully!


In [16]:
from datasets import load_dataset

dataset_name = "argilla/ultrafeedback-binarized-preferences"

try:
    # Load the training split of the dataset
    preference_dataset = load_dataset(dataset_name, split="train")

    print(f"Dataset '{dataset_name}' loaded successfully!")
    print(f"Number of examples: {len(preference_dataset)}")

    # Let's see the first entry to understand its structure
    print("\nFirst example entry:")
    print(preference_dataset[0])

    # You'll notice it has 'prompt', 'chosen', and 'rejected' fields,
    # which is exactly what DPOTrainer needs!
    print("\nKeys in the first example:", preference_dataset[0].keys())

except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure you have an active internet connection and the dataset name is correct.")



Dataset 'argilla/ultrafeedback-binarized-preferences' loaded successfully!
Number of examples: 63619

First example entry:
{'source': 'evol_instruct', 'instruction': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'chosen_response': 'Here\'s a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:\n\n#include <iostream>\n#include <string>\n#include <set>\n#include <map>\n#include <algorithm>\n\nusing namespace std;\n\nint main() {\n    // store countries and their bordering seas in a map\n    map<string, set<string>> countr

In [17]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [18]:
# 3. Define the corrected formatting function for DPOTrainer
def format_for_dpo(example):
    # Use the correct column names from the dataset
    user_message = example['instruction'] # Corrected: 'prompt' is actually 'instruction'
    chosen_response = example['chosen_response']
    rejected_response = example['rejected_response']

    # Format the user's message as the DPO prompt using Gemma's chat template
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Format the full chosen conversation (user + chosen assistant response)
    formatted_chosen = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": user_message},
            {"role": "model", "content": chosen_response}
        ],
        tokenize=False,
        add_generation_prompt=False
    )

    # Format the full rejected conversation (user + rejected assistant response)
    formatted_rejected = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": user_message},
            {"role": "model", "content": rejected_response}
        ],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "prompt": formatted_prompt,
        "chosen": formatted_chosen,
        "rejected": formatted_rejected,
    }

In [19]:
# 4. Apply the formatting function to your dataset
# We'll explicitly specify the columns to remove to avoid errors later
columns_to_remove = preference_dataset.column_names # Get all original columns
# Ensure 'prompt', 'chosen', 'rejected' are not in columns_to_remove if they were original,
# but our function is creating new ones, so it's fine.
# The map function will add 'prompt', 'chosen', 'rejected' and then remove the original ones.
processed_dataset = preference_dataset.map(
    format_for_dpo,
    num_proc=4,
    remove_columns=columns_to_remove # This will remove 'instruction', 'chosen_response', etc.
)

# Let's inspect a processed example to see the final format and column names
print("\n--- First Processed Example Entry ---")
print(processed_dataset[0])
print("\n--- Processed Dataset Column Names ---")
print(processed_dataset.column_names)

Map (num_proc=4):   0%|          | 0/63619 [00:00<?, ? examples/s]


--- First Processed Example Entry ---
{'prompt': '<bos><start_of_turn>user\nCan you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}<end_of_turn>\n<start_of_turn>model\n', 'chosen': '<bos><start_of_turn>user\nCan you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if countr

In [20]:
pip install trl accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 8.1 MB/s eta 0:00:00


In [21]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# We already have tokenizer from previous step, assuming model_name is "google/gemma-2b-it"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # NormalFloat 4-bit, a good default
    bnb_4bit_compute_dtype=torch.bfloat16, # Use bfloat16 for computation if your GPU supports it (NVIDIA Ampere architecture and newer)
    bnb_4bit_use_double_quant=False, # Optional double quantization
)

# Load the base model with quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # Automatically maps model layers to available devices (GPU/CPU)
)

# Prepare the model for k-bit training (important for QLoRA)
model.config.use_cache = False # Recommended for training
model.config.pretraining_tp = 1 # Gemma specific setting
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # Rank of the update matrices. A common value, experiment with 8, 16, 32, 64
    lora_alpha=16, # Scaling factor for LoRA updates. Usually double 'r'.
    lora_dropout=0.05, # Dropout percentage for LoRA layers
    bias="none", # Don't apply LoRA to bias terms
    task_type="CAUSAL_LM", # Specify that this is for causal language modeling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # These are the linear layers in Gemma where LoRA will be applied.
    # It's important to target key attention and feed-forward layers.
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

print("Model prepared for QLoRA DPO training!")
model.print_trainable_parameters() # This will show how few parameters you're actually training!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model prepared for QLoRA DPO training!
trainable params: 9,805,824 || all params: 2,515,978,240 || trainable%: 0.3897


In [24]:
# Load the reference model (crucially, define it BEFORE DPOTrainer)
# This model is frozen and used to calculate the DPO loss.
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16, # Match compute dtype if possible
    device_map="auto",
)
print("Reference Model loaded.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Reference Model loaded.


In [40]:
from trl import DPOConfig, DPOTrainer
import os

# Define the output directory for your trained model and logs
output_dir = "./gemma-2b-it-dpo-finetuned"
os.makedirs(output_dir, exist_ok=True) # Create the directory if it doesn't exist

training_args = DPOConfig(
    output_dir=output_dir, # Where to save the model and logs
    per_device_train_batch_size=1, # Adjust based on your GPU VRAM, start small (e.g., 2 or 4)
    gradient_accumulation_steps=8, # Accumulate gradients over multiple steps to simulate a larger batch size
    gradient_checkpointing=True, # Saves VRAM, but slightly slower
    learning_rate=5e-5, # Common learning rate for DPO, experiment with 1e-5 to 5e-5
    num_train_epochs=1, # Start with 1 epoch, DPO can converge quickly
    logging_steps=100, # Log training progress every 100 steps
    save_steps=500, # Save a checkpoint every 500 steps
    eval_steps=500, # Evaluate every 500 steps (if you have an eval set)
    save_total_limit=2, # Keep only the last 2 checkpoints
    report_to="tensorboard", # Log metrics to TensorBoard for visualization
    # DPO specific parameters:
    beta=0.1, # Critical DPO hyperparameter. Higher beta = more conservative (less deviation from ref model). Common range: 0.05 to 0.2
    max_length=512, # Max total length of prompt + completion
    max_prompt_length=256, # Max length of the prompt part
    ref_model_init_kwargs={"load_in_4bit": True}
    # truncation_mode="keep_end", # Default in DPOConfig, but good to be aware of
    # seed=42, # For reproducibility
    # You can add evaluation dataset here if you have one.
    # eval_dataset=processed_eval_dataset, # If you split your dataset into train/eval
)

In [41]:
# Ensure all previous setup code (model loading, tokenizer, processed_dataset, ref_model) is run before this

dpo_trainer = DPOTrainer(
    model=model, # Our QLoRA-enabled model
    ref_model=None, # Our frozen reference model
    args=training_args, # The DPOConfig we just defined
    train_dataset=processed_dataset, # Our prepared training dataset
    # For DPO, you often don't need a data_collator if max_length and max_prompt_length are set
    # as DPOTrainer uses its own default data collator which handles this.
)

/usr/local/lib/python3.11/dist-packages/trl/trainer/dpo_trainer.py:298: UserWarning: You passed ref_model_init_kwargs to the `DPOConfig`, but your ref_model is already instantiated. The `ref_model_init_kwargs` will be ignored.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/63619 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [42]:
# Start Training!
print("\nStarting DPO training...")
dpo_trainer.train()
print("DPO training complete!")

# Save the Merged Model
print(f"\n--- Merging LoRA adapters and saving the final model to {output_dir}/final_merged_model ---")
final_output_dir = os.path.join(output_dir, "final_merged_model")
os.makedirs(final_output_dir, exist_ok=True)

# First, save the PEFT adapters
dpo_trainer.model.save_pretrained(final_output_dir)

# Then, load the base model again and merge the adapters.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # or torch.bfloat16
    device_map="auto"
)


Starting DPO training...


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 1000.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 726.12 MiB is free. Process 76279 has 14.03 GiB memory in use. Of the allocated memory 13.21 GiB is allocated by PyTorch, and 703.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Load the PEFT adapters onto the base model
merged_model = PeftModel.from_pretrained(base_model_for_merge, final_output_dir)

# Merge and unload the adapters
merged_model = merged_model.merge_and_unload()

# Save the merged model and tokenizer
merged_model.save_pretrained(final_output_dir, safe_serialization=True)
tokenizer.save_pretrained(final_output_dir)

print("Final merged model and tokenizer saved! Your DPO fine-tuning is complete.")